**This part is required for all the notebook**

In [1]:
import sys
sys.path.append("lib")

from main import Notebook, HandsOn1CatsnifferUI

nb = Notebook()

# Notes:
# Fix typing error:
# pip install --upgrade typing_extensions

# NOTE:
This part of the notebook requires internet connection, please run only once.

# Requirements
## Clone or update Catsniffer-Tools
This code cell will execute part of the CatSniffer setup process. It checks out or updates the CatSniffer repository containing firmware and utilities.

> **Please run the following code block before begin the Labs!**

In [ ]:
nb.clone_catsniffer_tools()

## Installing the Requierements
This code cell will execute part of the CatSniffer Catnip tool to download the last firmware from the repo.

In [ ]:
nb.install_catsniffer_requirements()

## Download Firmware files
This code cell will execute part of the CatSniffer Catnip tool process. It checks out and download the CatSniffer firmware.

In [ ]:
nb.download_catsniffer_firmware()

---

# Hands-On Lab #1: Catsniffer + Meshtastic

## Objectives
- Explain the role of CatSniffer in capturing and analyzing 915 MHz RF signals.
- Configure CatSniffer as a LoRa sniffer for Meshtastic traffic.
- Identify and isolate Meshtastic communication patterns in the RF spectrum.
- Use the Meshtastic decrypter with known PSKs to extract public messages.
- Use the Meshtastic live decoder to parse and understand packet payloads.

## Challenge
Flash the CatSniffer with the LoRa sniffer firmware. Monitor real-time activity in the 900 MHz band to identify Meshtastic node transmissions.
Distinguish Meshtastic packets from other traffic based on RF characteristics.
Capture encrypted payloads, apply known keys using the Meshtastic decrypter, and extract readable messages. Use the live decoder to analyze packet structure and content.

## Walk-Through

First of all we need to flash our LoRa Sniffer for the RP2040 since this is the microcontroller who has access to the SX1262 by SPI.
![Catsniffer Components Diagram](static/block_diagrams.png)

### Flash firmware

> Run the next code, and after that follow the steps to put the bootloader mode


In [ ]:
serialpass_file = "lorasniffer.uf2"
nb.flash_uf2_firmware(serialpass_file)


### RP2040 bootloader mode:

![Catsniffer Board](static/banner.jpg)


- Press and hold boot RP2040
- While Holding Boot RP2040, press Reset RP2040
- The device will be shown as a thumbdrive

![RP 2040 Boot](static/RP_USB.png)



For this laboratory we are going to try to catch messages from the meshtastic firmware for Black Hat EU. If we dig a bit in the firmware repository for meshtastic firmware we can find the jsonc file used to configure any node once programmed.

### Radio Configuration

Radio configurations for this meshtastic network are:

| Parameter        | Value      | Command         |
|------------------|------------|-----------------|
| Frequency        | 906.875 MHz | set_freq 906.875 |
| Bandwidth        | 500 KHz    | set_bw 9        |
| Spreading Factor | 7          | set_sf 7        |
| Coding Rate      | 4/5        | set_cr 5        |
| Sync Word        | 0x2B       | set_sw 0x2B     |
| Preamble Length  | 8         | set_pl 8       |

If these radio configurations are not equal, you wont be able to receive any data from this mesh.

### Configuring the radio
- **Configuration commands**:
    - set_rx
    - set_tx
    - set_tx_hex
    - set_tx_ascii
    - set_region
    - set_chann
    - set_freq
    - set_sf
    - set_bw
    - set_cr
    - set_sw
    - set_pl
    - set_op
- **Monitor commands**:
    - get_freq
    - get_sf
    - get_bw
    - get_cr
    - get_sw
    - get_pl
    - get_op
    - get_config

We use the commands to setup the radio:
- `set_freq 906.875`
- `set_bw 8`
- `set_sf 11`
- `set_pl 8`
- `set_sw 0x2B`
- `set_rx`

In [ ]:
lab1ui = HandsOn1CatsnifferUI()
lab1ui.display_ui_terminal()


Then we will be able to receive data from the mesh network. Now lets open up our sniffer tool for wireshark.

### Running with Wireshark

In [ ]:
lab1ui.display_ui_wireshark()

![Wireshark Meshtastic Packet](static/ws_meshtastic.png)

You should be able to read the lora payload that can be decoded with the python script `meshtasticDecoder.py`

### Decoding Telemetry

```bash
python meshtasticDecoder.py -k EXTRACTED_KEY -i PAYLOAD
```

In [ ]:
lab1ui.display_ui_cmd_decode_tm()

### Live Telemetry decoding
Once decrypted it needs to be decoded by protobuf, so we finally get a *TELEMETRY* message.


But this is too slow, so we can move on to the python script, `meshtasticLiveDecoder.py`

```bash
python meshtasticLiveDecoder.py -p /dev/ttyACM0 -baud 921600 -f 917.25 -ps ShortFast
```

In [ ]:
lab1ui.display_ui_live_decoding()